<a href="https://colab.research.google.com/github/Innovatewithapple/CNNProjects/blob/main/PyTorchCNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Images → Transform → Dataset → DataLoader → CNN → Loss → Optimizer → Training Loop → Validation**

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [2]:
train_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
])

val_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
])

In [3]:
train_dataset = datasets.ImageFolder(
    root="data/train",
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    root="data/val",
    transform=val_transform
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

FileNotFoundError: [Errno 2] No such file or directory: 'data/train'

In [4]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),   # (128x128) → (128x128)
            nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),                  # → (64x64)

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),                  # → (32x32)
        )

        self.fc = nn.Sequential(
            nn.Linear(64 * 32 * 32, 128),
            nn.ReLU(),
            nn.Linear(128, 1)  # binary classification
        )

    def forward(self, x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)  # flatten
        x = self.fc(x)
        return x

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CNN().to(device)

loss_fn = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [8]:
epochs = 10

for epoch in range(epochs):

    # 🔵 TRAIN
    model.train()
    train_loss = 0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = loss_fn(outputs, labels.float().unsqueeze(1))

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

        preds = (torch.sigmoid(outputs) > 0.5).int()
        train_correct += (preds.squeeze() == labels).sum().item()
        train_total += labels.size(0)

    train_acc = train_correct / train_total

    # 🟢 VALIDATION
    model.eval()
    val_loss = 0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = loss_fn(outputs, labels.float().unsqueeze(1))

            val_loss += loss.item()

            preds = (torch.sigmoid(outputs) > 0.5).int()
            val_correct += (preds.squeeze() == labels).sum().item()
            val_total += labels.size(0)

    val_acc = val_correct / val_total

    print(f"""
    Epoch {epoch+1}
    Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}
    Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f}
    """)

NameError: name 'train_loader' is not defined